# BenchMARL — Multi-Algorithm × Multi-Seed Runner
Chạy thực nghiệm MARL trên Kaggle / Google Colab.

## 1. Cài đặt

In [ ]:
# ── Kiểm tra GPU ────────────────────────────────────────────────────────────
!nvidia-smi 2>/dev/null || echo 'No GPU detected, running on CPU'

# ── Cài đặt xvfb (headless display) ────────────────────────────────────────
!apt-get install -y xvfb 2>/dev/null | tail -1

# ── Cài đặt BenchMARL và dependencies ──────────────────────────────────────
# Option A: cài từ repo local (nếu đã clone)
import os, sys
REPO_PATH = '/kaggle/working/BenchMARL'   # thay đổi nếu cần
os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH)
!pip install -e . -q

# ── Cài vmas (task environment) ─────────────────────────────────────────────
!pip install vmas -q

print('\n✓ Installation done')

## 2. Config thực nghiệm

In [ ]:
# ════════════════════════════════════════════════════════════
#  CHỈNH SỬA Ở ĐÂY
# ════════════════════════════════════════════════════════════

ALGORITHMS = [
    'mappo',
    'ippo',
    'ippo_vi',
    'ippo_extragradient',
    'ippo_vi_no_norm',
    'ippo_vi_no_anchor',
]

TASKS = [
    'vmas/balance',
    # 'vmas/navigation',
]

SEEDS = [0, 1, 2]

# Overrides Hydra — điều chỉnh hyperparameter tại đây
EXPERIMENT_OVERRIDES = [
    'experiment.loggers=[csv]',
    'experiment.render=false',
    'experiment.max_n_frames=3_000_000',
    'experiment.evaluation_interval=120_000',
    'experiment.on_policy_collected_frames_per_batch=6000',
    'experiment.on_policy_n_envs_per_worker=10',
    'experiment.on_policy_minibatch_size=400',
    'experiment.on_policy_n_minibatch_iters=45',
    'experiment.lr=0.00005',
    'experiment.checkpoint_interval=0',
]

print(f'Algorithms : {ALGORITHMS}')
print(f'Tasks      : {TASKS}')
print(f'Seeds      : {SEEDS}')
print(f'Total runs : {len(ALGORITHMS) * len(TASKS) * len(SEEDS)}')

## 3. Run (Hydra --multirun) — **khuyên dùng**

In [ ]:
import subprocess, sys, itertools
from pathlib import Path

REPO_ROOT = Path(REPO_PATH)

algo_str = ','.join(ALGORITHMS)
task_str = ','.join(TASKS)
seed_str = ','.join(map(str, SEEDS))
overrides = ' '.join(EXPERIMENT_OVERRIDES)

cmd = (
    f"xvfb-run -a python benchmarl/run.py "
    f"--multirun "
    f"algorithm={algo_str} "
    f"task={task_str} "
    f"seed={seed_str} "
    f"{overrides}"
)

print('CMD:', cmd)
!{cmd}

## 4. Run (sequential loop) — dễ debug hơn

In [ ]:
import subprocess, sys, itertools, shlex
from pathlib import Path

REPO_ROOT = Path(REPO_PATH)
failed = []

combos = list(itertools.product(ALGORITHMS, TASKS, SEEDS))
print(f'Total: {len(combos)} runs\n')

for idx, (algo, task, seed) in enumerate(combos, 1):
    tag = f'[{algo} | {task} | seed={seed}]'
    print(f'\n[{idx}/{len(combos)}] START {tag}')

    overrides_str = ' '.join(EXPERIMENT_OVERRIDES)
    cmd = (
        f"xvfb-run -a python benchmarl/run.py "
        f"algorithm={algo} task={task} seed={seed} "
        f"{overrides_str}"
    )

    result = subprocess.run(cmd, shell=True, cwd=REPO_ROOT)

    if result.returncode == 0:
        print(f'✓ DONE  {tag}')
    else:
        print(f'✗ FAIL  {tag}  (exit={result.returncode})')
        failed.append((algo, task, seed))

print('\n' + '='*60)
if failed:
    print(f'{len(failed)} run(s) THẤT BẠI:')
    for f in failed:
        print(f'  - {f}')
else:
    print('Tất cả runs hoàn thành ✓')

## 5. Xem kết quả CSV

In [ ]:
import pandas as pd
from pathlib import Path

# BenchMARL lưu CSV vào outputs/ hoặc multirun/ (thư mục Hydra)
# Tìm tất cả file info.json để xác định các run
result_dirs = list(Path(REPO_ROOT).rglob('*.csv'))
print(f'Found {len(result_dirs)} CSV file(s):')
for f in result_dirs:
    print(' ', f.relative_to(REPO_ROOT))

In [ ]:
# ── Đọc và tổng hợp kết quả ─────────────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

dfs = []
for csv_path in Path(REPO_ROOT).rglob('*evaluation*.csv'):
    # Lấy thông tin algo/task/seed từ đường dẫn hydra
    parts = csv_path.parts
    try:
        df = pd.read_csv(csv_path)
        df['source'] = str(csv_path.parent.relative_to(REPO_ROOT))
        dfs.append(df)
    except Exception as e:
        print(f'Skip {csv_path}: {e}')

if dfs:
    results = pd.concat(dfs, ignore_index=True)
    print(results.head())
else:
    print('Chưa có kết quả CSV.')